In [11]:
# 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
"""
RAG 검색 결과를 평가하기 위한 검색 평가 지표 모음.

평가 지표:
- Context recall: 정답을 만드는 데 필요한 정보가 검색된 청킹 안에
  누락 없이 들어있는지 평가한다.
- Context precision: 검색된 청킹들 중 진짜 답변에 필요한 정보가
  상위에 잘 배치되어 있는지 평가한다.
- MRR: 사용자가 원하는 첫 번째 정답 조각이 얼마나 맨 위에 노출되었는지
  순위 기반으로 평가한다.
- nDCG: 검색된 문서들의 관련도 점수와 순서를 모두 고려해 평가한다.
  가장 관련성이 높은 고품질 청킹이 맨 위에 올수록 높은 점수를 준다.

입력 컬럼 예시:
question_id, question, ground_truth, retrieved_contexts

`retrieved_contexts`는 다음 형식을 지원한다:
- 문자열 리스트
- JSON 또는 Python literal 형태의 리스트 문자열
- "\\n---\\n", "|||", "\\n\\n" 중 하나로 구분된 단일 문자열
- retrieved_context_1, retrieved_context_2, ... 형태의 다중 컬럼
"""

from __future__ import annotations

import argparse
import ast
import json
import math
import re
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence


DEFAULT_GROUND_TRUTH_ROWS = [
    {
        "question_id": 1,
        "question": "고려대학교 차세대 포털·학사 정보시스템 구축 사업의 총 사업예산(V.A.T 포함)과 계약일로부터의 사업 기간은 각각 어떻게 명시되어 있나요?",
        "ground_truth": "총 사업예산은 부가가치세 포함 11,270,000,000원이며 사업 기간은 계약일로부터 24개월 이내입니다.",
    },
    {
        "question_id": 2,
        "question": "본 사업의 예산 집행 계획 중 2025학년도에 지급될 예정인 사업비의 비율은 전체의 몇 퍼센트인가요?",
        "ground_truth": "2025학년도에는 전체 사업예산의 약 40%를 분할 지급할 예정입니다.",
    },
    {
        "question_id": 3,
        "question": "성능요구사항(PER-005)에 명시된 기준에 따라 수강신청 시스템과 같이 특정 시간에 사용자가 폭증하는 시스템의 경우 보장해야 하는 최소 동시 사용자 수는 얼마인가요?",
        "ground_truth": "수강신청 시스템 등 트래픽 폭증 시스템의 경우 동시 사용자 15,000명 이상을 지원해야 합니다.",
    },
    {
        "question_id": 4,
        "question": "성능요구사항인 온라인성 업무응답시간(PER-002)과 웹페이지 디스플레이시간(PER-003)에서 목표로 하는 처리 시간은 각각 요청 후 몇 초 이내인가요?",
        "ground_truth": "모든 질의에 대한 결과 처리 및 웹페이지의 완전한 출력은 사용자가 요청한 시간으로부터 3초 내에 완료되어야 합니다.",
    },
    {
        "question_id": 5,
        "question": "상세 요구사항 분류기준에 따른 본 사업의 총 요구사항 수와 그 중 가장 많은 비중을 차지하는 기능요구사항(SFR)의 개수는 각각 얼마인가요?",
        "ground_truth": "본 사업의 총 요구사항 수는 160개이며 이 중 기능요구사항(SFR)은 99개입니다.",
    },
    {
        "question_id": 6,
        "question": "기능요구사항 ID SFR-포털-009 지능형 검색에서 사용자 의도와 요구에 맞게 도입해야 하는 4가지 핵심 검색 방식은 무엇인가요?",
        "ground_truth": "의미기반 검색, 개인화 검색 / 추천 검색, 유사문장 검색, 다국어 검색의 4가지 방식을 도입해야 합니다.",
    },
    {
        "question_id": 7,
        "question": "제안 평가 방식 중 기술평가와 가격평가의 배점 비율은 각각 어떻게 구성되며 기술평가 항목 중 기술 및 기능 부문에 배정된 점수는 몇 점인가요?",
        "ground_truth": "기술평가 90%와 가격평가 10%로 구성되며 기술평가 항목 중 기술 및 기능 부문에는 30점이 배정되어 있습니다.",
    },
    {
        "question_id": 8,
        "question": "표준요구사항 STR-001에 따라 웹 호환성 및 접근성 준수를 위해 만족해야 하는 보안 가이드의 기준 연도와 최종 제출해야 하는 산출물 보고서의 명칭은 무엇인가요?",
        "ground_truth": "행정안전부의 소프트웨어 개발보안 가이드(2021.11) 기준을 만족해야 하며 웹 접근성 결과 보고서를 제출해야 합니다.",
    },
    {
        "question_id": 9,
        "question": "기능요구사항 SFR-학사-013 수강소감관리에서 강의 개선을 위해 관리자가 작성해야 하는 보고서 명칭과 해당 기능에서 종합적으로 조회해야 하는 4가지 이력 항목은 무엇인가요?",
        "ground_truth": "강의개선보고서(CQI)를 작성해야 하며 CQI보고서 내용, 수강소감설문 사후조치 내용, 강의평가 이력, 역량수준 이력을 종합 조회해야 합니다.",
    },
    {
        "question_id": 10,
        "question": "기능요구사항 SFR-모바일-001에 명시된 지침에 따라 기존 모바일 통합앱인 호잇에서 흡수하여 재구축해야 하는 주요 기능 5가지를 기술하세요.",
        "ground_truth": "검색, 교내연락처, 푸시, 일정, 유니버스가 포함되며 이 외에도 식단, 교통서비스 등을 흡수하여 재구축해야 합니다.",
    },
]

STOPWORDS = {
    "은", "는", "이", "가", "을", "를", "에", "의", "와", "과", "및", "등",
    "그", "중", "본", "각각", "어떻게", "무엇인가요", "얼마인가요", "기술하세요",
    "합니다", "입니다", "되어", "있는", "위해", "경우",
}


def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_token(token: str) -> str:
    """검색 overlap 평가를 위한 간단한 한국어 조사 정규화."""
    token = token.lower()
    if re.fullmatch(r"[가-힣]+", token) and len(token) > 2:
        for suffix in (
            "으로부터", "에서는", "에게는", "에는", "에서", "으로",
            "로서", "로써", "은", "는", "이", "가", "을", "를", "의", "와", "과", "로",
        ):
            if token.endswith(suffix) and len(token) > len(suffix) + 1:
                return token[: -len(suffix)]
    return token


def tokenize(text: Any, *, remove_stopwords: bool = True) -> List[str]:
    text = normalize_text(text).lower()
    tokens = [
        normalize_token(token)
        for token in re.findall(r"[가-힣]+|[a-zA-Z]+|\d+(?:[.,]\d+)*%?", text)
    ]
    if not remove_stopwords:
        return tokens
    return [token for token in tokens if token not in STOPWORDS and len(token) > 1]


def safe_div(numerator: float, denominator: float) -> float:
    return numerator / denominator if denominator else 0.0


def _split_context_string(value: str) -> List[str]:
    if value is None:
        return []
    original = str(value).strip()
    if not original or original.lower() == "nan":
        return []
    for separator in ("\n---\n", "|||", "\n\n"):
        if separator in original:
            return [normalize_text(part) for part in original.split(separator) if normalize_text(part)]
    return [normalize_text(original)]


def parse_retrieved_contexts(
    row: Dict[str, Any],
    contexts_col: str = "retrieved_contexts",
    context_prefixes: Sequence[str] = (
        "retrieved_context_",
        "retrieved_chunk_",
        "context_",
        "chunk_",
    ),
) -> List[str]:
    """일반적인 DataFrame/CSV 구조에서 순위가 있는 검색 청크를 추출한다."""
    value = row.get(contexts_col)
    if isinstance(value, list):
        return [normalize_text(item) for item in value if normalize_text(item)]

    if value is not None and normalize_text(value):
        raw = normalize_text(value)
        if raw.lower() != "nan":
            try:
                parsed = json.loads(raw)
            except (TypeError, ValueError):
                try:
                    parsed = ast.literal_eval(raw)
                except (SyntaxError, ValueError):
                    parsed = None

            if isinstance(parsed, list):
                return [normalize_text(item) for item in parsed if normalize_text(item)]
            return _split_context_string(str(value))

    ranked_columns = []
    for col in row:
        for prefix in context_prefixes:
            if col.startswith(prefix):
                suffix = col.removeprefix(prefix)
                rank = int(suffix) if suffix.isdigit() else 10_000
                ranked_columns.append((rank, col))
                break

    contexts = []
    for _, col in sorted(ranked_columns):
        context = normalize_text(row.get(col, ""))
        if context and context.lower() != "nan":
            contexts.append(context)
    return contexts


def compute_context_recall(ground_truth: str, contexts: Sequence[str]) -> float:
    """전체 검색 청크가 정답 정보 토큰을 얼마나 포함하는지 계산한다."""
    gt_counts = Counter(tokenize(ground_truth))
    if not gt_counts:
        return 0.0
    context_tokens = set(tokenize(" ".join(contexts)))
    covered = sum(count for token, count in gt_counts.items() if token in context_tokens)
    return round(safe_div(covered, sum(gt_counts.values())), 4)


def compute_chunk_relevance(ground_truth: str, context: str) -> float:
    """
    정답과 검색 청크의 토큰 overlap을 기반으로 청크 관련도 점수를 계산한다.

    검색 청크는 답변에 필요한 정보 외의 주변 문맥도 함께 포함할 수 있으므로,
    precision보다 recall에 조금 더 높은 가중치를 둔다.
    """
    gt_tokens = tokenize(ground_truth)
    context_tokens = tokenize(context)
    if not gt_tokens or not context_tokens:
        return 0.0

    gt_counts = Counter(gt_tokens)
    context_counts = Counter(context_tokens)
    overlap = sum((gt_counts & context_counts).values())
    recall = safe_div(overlap, len(gt_tokens))
    precision = safe_div(overlap, len(context_tokens))
    relevance = (0.7 * recall) + (0.3 * precision)
    return round(min(relevance, 1.0), 4)


def compute_context_precision(
    relevance_scores: Sequence[float],
    relevance_threshold: float = 0.2,
) -> float:
    """
    검색 청크의 Average Precision을 계산한다.

    청크 관련도 점수가 `relevance_threshold` 이상이면 답변에 유용한 청크로 보고,
    유용한 청크가 상위에 있을수록 높은 점수를 준다.
    """
    relevant_count = 0
    precision_sum = 0.0
    for rank, score in enumerate(relevance_scores, start=1):
        if score >= relevance_threshold:
            relevant_count += 1
            precision_sum += safe_div(relevant_count, rank)
    return round(safe_div(precision_sum, relevant_count), 4)


def compute_mrr(
    relevance_scores: Sequence[float],
    relevance_threshold: float = 0.2,
) -> float:
    for rank, score in enumerate(relevance_scores, start=1):
        if score >= relevance_threshold:
            return round(1 / rank, 4)
    return 0.0


def _dcg(scores: Sequence[float]) -> float:
    return sum(
        ((2**score) - 1) / math.log2(rank + 1)
        for rank, score in enumerate(scores, start=1)
    )


def compute_ndcg(relevance_scores: Sequence[float], k: Optional[int] = None) -> float:
    if k is not None:
        relevance_scores = relevance_scores[:k]
    if not relevance_scores:
        return 0.0
    ideal_scores = sorted(relevance_scores, reverse=True)
    ideal_dcg = _dcg(ideal_scores)
    return round(safe_div(_dcg(relevance_scores), ideal_dcg), 4)


def evaluate_retrieval_row(
    row: Dict[str, Any],
    question_col: str = "question",
    ground_truth_col: str = "ground_truth",
    contexts_col: str = "retrieved_contexts",
    relevance_threshold: float = 0.2,
    ndcg_k: Optional[int] = None,
) -> Dict[str, Any]:
    contexts = parse_retrieved_contexts(row, contexts_col=contexts_col)
    ground_truth = normalize_text(row.get(ground_truth_col, ""))
    question = normalize_text(row.get(question_col, ""))
    relevance_scores = [compute_chunk_relevance(ground_truth, context) for context in contexts]

    first_relevant_rank = 0
    for rank, score in enumerate(relevance_scores, start=1):
        if score >= relevance_threshold:
            first_relevant_rank = rank
            break

    return {
        "context_recall": compute_context_recall(ground_truth, contexts),
        "context_precision": compute_context_precision(relevance_scores, relevance_threshold),
        "mrr": compute_mrr(relevance_scores, relevance_threshold),
        "ndcg": compute_ndcg(relevance_scores, ndcg_k),
        "retrieved_count": len(contexts),
        "first_relevant_rank": first_relevant_rank,
        "chunk_relevance_scores": ", ".join(f"{score:.4f}" for score in relevance_scores),
        "question_for_eval": question,
    }


def evaluate_retrieval_dataframe(
    df,
    question_col: str = "question",
    ground_truth_col: str = "ground_truth",
    contexts_col: str = "retrieved_contexts",
    relevance_threshold: float = 0.2,
    ndcg_k: Optional[int] = None,
):
    import pandas as pd

    scores = []
    for _, row in df.iterrows():
        scores.append(
            evaluate_retrieval_row(
                row.to_dict(),
                question_col=question_col,
                ground_truth_col=ground_truth_col,
                contexts_col=contexts_col,
                relevance_threshold=relevance_threshold,
                ndcg_k=ndcg_k,
            )
        )
    score_df = pd.DataFrame(scores)
    return pd.concat([df.reset_index(drop=True), score_df], axis=1)


def summarize_retrieval(
    evaluated_df,
    group_col: Optional[str] = "strategy",
    metric_cols: Optional[Sequence[str]] = None,
):
    metric_cols = list(
        metric_cols
        or ["context_recall", "context_precision", "mrr", "ndcg", "retrieved_count"]
    )
    if group_col and group_col in evaluated_df.columns:
        return (
            evaluated_df.groupby(group_col)[metric_cols]
            .mean(numeric_only=True)
            .round(4)
            .reset_index()
        )
    return evaluated_df[metric_cols].mean(numeric_only=True).round(4).to_frame("mean").T


# =====================================================
# [신규] B모드 전용 — 이미지 표 형식 터미널 출력
# =====================================================

def print_summary_table(summary_df, metric_cols=None) -> None:
    """
    summarize_retrieval() 결과를 이미지 표와 동일한 형식으로 터미널에 출력한다.
    각 지표의 최고값에 * 마커를 표시한다.
    """
    if metric_cols is None:
        metric_cols = ["context_recall", "context_precision", "mrr", "ndcg"]

    # strategy 컬럼이 없으면 index를 전략명으로 사용
    if "strategy" in summary_df.columns:
        df = summary_df.set_index("strategy")[metric_cols]
    else:
        df = summary_df[metric_cols]

    col_labels = {
        "context_recall":    "context_recall",
        "context_precision": "context_precision",
        "mrr":               "MRR",
        "ndcg":              "nDCG",
    }
    headers = [col_labels.get(c, c) for c in metric_cols]

    # 컬럼별 최고값
    best = {c: df[c].max() for c in metric_cols}

    strategy_w = max(len(str(idx)) for idx in df.index) + 2
    strategy_w = max(strategy_w, len("전략") + 2)
    col_w = 19

    total_w = strategy_w + col_w * len(metric_cols) + 2
    sep = "─" * total_w

    print()
    print("=" * total_w)
    print("  RAG Retrieval 전략별 평가 결과 요약 (B모드 / 토큰 overlap)")
    print("=" * total_w)
    header_row = f"  {'전략':<{strategy_w}}" + "".join(f"{h:>{col_w}}" for h in headers)
    print(header_row)
    print(sep)

    for strategy, row in df.iterrows():
        cells = []
        for c in metric_cols:
            val = row[c]
            marker = " *" if val == best[c] else "  "
            cells.append(f"{val:.4f}{marker}")
        row_str = f"  {str(strategy):<{strategy_w}}" + "".join(f"{cell:>{col_w}}" for cell in cells)
        print(row_str)

    print(sep)
    print("  * 각 지표 최고값")
    print("=" * total_w)
    print()


def make_default_ground_truth_dataframe():
    import pandas as pd
    return pd.DataFrame(DEFAULT_GROUND_TRUTH_ROWS)


def _load_input(path: str, sheet: Any = 0):
    import pandas as pd
    if path.lower().endswith(".xlsx"):
        sheet_name = int(sheet) if str(sheet).isdigit() else sheet
        return pd.read_excel(path, sheet_name=sheet_name)
    return pd.read_csv(path)


# =====================================================
# SECTION A: chunk_id 이진 Relevance 지표
#   rfp_rag_pipeline.py의 calc_* 함수와 완전히 동일한 로직.
#   Pipeline JSON 모드에서 chunk_id 기반 평가에 사용한다.
# =====================================================

def calc_precision(retrieved: list, relevant: set) -> float:
    if not retrieved:
        return 0.0
    return round(sum(1 for r in retrieved if r in relevant) / len(retrieved), 4)


def calc_recall(retrieved: list, relevant: set) -> float:
    if not relevant:
        return 0.0
    return round(sum(1 for r in retrieved if r in relevant) / len(relevant), 4)


def calc_mrr(retrieved: list, relevant: set) -> float:
    for rank, r in enumerate(retrieved, start=1):
        if r in relevant:
            return round(1.0 / rank, 4)
    return 0.0


def calc_ndcg(retrieved: list, relevant: set, k: int) -> float:
    def dcg(ids):
        return sum(
            1.0 / math.log2(i + 2)
            for i, cid in enumerate(ids[:k])
            if cid in relevant
        )
    actual = dcg(retrieved)
    ideal  = dcg(list(relevant)[:k])
    return round(safe_div(actual, ideal), 4)


# =====================================================
# SECTION B: Pipeline JSON 평가 (A모드 전용 / 참고용)
# =====================================================

def evaluate_from_pipeline_json(
    json_path: str,
    ndcg_k: Optional[int] = None,
) -> Dict[str, Any]:
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    config    = data.get("config", {})
    per_query = data.get("per_query", [])
    k         = ndcg_k or config.get("final_top_k", 5)

    enriched = []
    for row in per_query:
        retrieved    = row.get("retrieved_ids", [])
        relevant_set = set(row.get("relevant_ids", []))

        precision = row["precision"] if "precision" in row else calc_precision(retrieved, relevant_set)
        recall    = row["recall"]    if "recall"    in row else calc_recall(retrieved, relevant_set)
        mrr_val   = row["mrr"]       if "mrr"       in row else calc_mrr(retrieved, relevant_set)
        ndcg_val  = row["ndcg"]      if "ndcg"      in row else calc_ndcg(retrieved, relevant_set, k)

        first_hit_rank = next(
            (i + 1 for i, cid in enumerate(retrieved) if cid in relevant_set), 0
        )

        enriched.append({
            **row,
            "precision":      round(float(precision), 4),
            "recall":         round(float(recall),    4),
            "mrr":            round(float(mrr_val),   4),
            "ndcg":           round(float(ndcg_val),  4),
            "first_hit_rank": first_hit_rank,
        })

    def _avg(key: str) -> float:
        vals = [r[key] for r in enriched if r.get(key) is not None]
        return round(sum(vals) / len(vals), 4) if vals else 0.0

    averages = {
        "context_precision": _avg("precision"),
        "context_recall":    _avg("recall"),
        "mrr":               _avg("mrr"),
        f"ndcg@{k}":         _avg("ndcg"),
    }

    return {"config": config, "averages": averages, "per_query": enriched}


def print_pipeline_results(result: Dict[str, Any]) -> None:
    config = result["config"]
    rows   = result["per_query"]
    k      = config.get("final_top_k", 5)

    Q = 52
    header = (
        f"{'No':>3}  {'질문':<{Q}}  "
        f"{'Precision':>9}  {'Recall':>6}  {'MRR':>6}  {f'NDCG@{k}':>7}  "
        f"{'관련/검색':>7}  {'첫 Hit':>6}"
    )
    sep = chr(9472) * len(header)

    print()
    print("=" * len(header))
    print("  RFP RAG Retrieval 평가 결과 (retrieval.py)")
    print("=" * len(header))
    print(f"  PDF          : {config.get('pdf', '-')}")
    print(f"  Embed Model  : {config.get('embed_model', '-')}")
    print(f"  FAISS top_k  : {config.get('faiss_top_k', '-')}")
    if config.get("rerank"):
        print(f"  Reranker     : {config.get('rerank_model', '-')}  top_k={config.get('rerank_top_k')}")
    print(f"  총 청크 수   : {config.get('num_chunks', '-')}")
    print("-" * len(header))
    print(header)
    print(sep)

    for r in rows:
        q_short = r["question"][:Q - 1] + "..." if len(r["question"]) > Q else r["question"]
        hits  = r.get("relevant_hits", 0)
        total = r.get("retrieved_count", 0)
        fhit  = r.get("first_hit_rank", 0)
        print(
            f"{r['num']:>3}  {q_short:<{Q}}  "
            f"{r['precision']:>9.4f}  {r['recall']:>6.4f}  "
            f"{r['mrr']:>6.4f}  {r['ndcg']:>7.4f}  "
            f"{hits:>3}/{total:<3}  "
            f"{'#'+str(fhit) if fhit else '-':>6}"
        )

    print(sep)
    avgs     = result["averages"]
    ndcg_key = f"ndcg@{k}"
    print(
        f"{'평균':>3}  {f'(전체 {len(rows)}개 질문)':.<{Q}}  "
        f"{avgs['context_precision']:>9.4f}  {avgs['context_recall']:>6.4f}  "
        f"{avgs['mrr']:>6.4f}  {avgs[ndcg_key]:>7.4f}"
    )
    print("=" * len(header))
    print()


# =====================================================
# main(): B모드 전용 진입점
# =====================================================

def main() -> int:
    parser = argparse.ArgumentParser(description="RAG 검색 결과 CSV/XLSX 평가 (B모드 전용)")
    parser.add_argument("--input",     required=True,  help="검색 결과 파일 경로 (csv/xlsx)")
    parser.add_argument("--output",    default="retrieval_eval_result.csv",  help="행별 평가 결과 저장 경로")
    parser.add_argument("--summary",   default="retrieval_eval_summary.csv", help="전략별 요약 저장 경로")
    parser.add_argument("--sheet",     default=0,       help="xlsx 시트 이름 또는 index")
    parser.add_argument("--question-col",        default="question")
    parser.add_argument("--ground-truth-col",    default="ground_truth")
    parser.add_argument("--contexts-col",        default="retrieved_contexts")
    parser.add_argument("--relevance-threshold", type=float, default=0.2)
    parser.add_argument("--ndcg-k",    type=int, default=None, help="NDCG@K의 K값 (기본: 전체)")
    parser.add_argument("--group-col", default="strategy", help="전략 그룹 컬럼명 (기본: strategy)")
    parser.add_argument(
        "--write-default-gt",
        default=None,
        help="내장 ground truth 10개를 CSV로 저장할 경로 (단독 실행)",
    )
    args = parser.parse_args()

    # [버그 수정 1] --write-default-gt 실행 후 즉시 종료 (원본: return 0 누락)
    if args.write_default_gt:
        gt_df = make_default_ground_truth_dataframe()
        gt_df.to_csv(args.write_default_gt, index=False, encoding="utf-8-sig")
        print("기본 ground truth 저장 완료:", args.write_default_gt)
        return 0  # ← 원본에서 누락

    # [버그 수정 2] args.pipeline_json 참조 제거
    #   원본 코드에서 argparse에 등록하지 않은 args.pipeline_json을
    #   참조하여 AttributeError 발생 → B모드에서 해당 분기 전체 삭제

    # B 모드: CSV/XLSX 토큰 overlap 평가
    input_df = _load_input(args.input, args.sheet)
    evaluated = evaluate_retrieval_dataframe(
        input_df,
        question_col=args.question_col,
        ground_truth_col=args.ground_truth_col,
        contexts_col=args.contexts_col,
        relevance_threshold=args.relevance_threshold,
        ndcg_k=args.ndcg_k,
    )
    summary = summarize_retrieval(evaluated, group_col=args.group_col)

    # 이미지 표 형식으로 터미널 출력
    print_summary_table(summary)

    # CSV 저장
    evaluated.to_csv(args.output,  index=False, encoding="utf-8-sig")
    summary.to_csv(args.summary,   index=False, encoding="utf-8-sig")
    print("행별 평가 결과 저장 완료 :", args.output)
    print("전략별 요약 저장 완료   :", args.summary)
    return 0

In [13]:
def run_eval(input_file: str, ndcg_k: Optional[int] = 3) -> int:
    """
    코랩 셀에서 바로 실행하기 위한 커스텀 진입점입니다.
    --ndcg-k의 기본값을 사용자 요청에 맞춰 3으로 세팅했습니다.
    """
    import pandas as pd

    output_path = "retrieval_eval_result.csv"
    summary_path = "retrieval_eval_summary.csv"

    # 데이터 로드
    input_df = _load_input(input_file, 0)

    # 평가 수행
    evaluated = evaluate_retrieval_dataframe(
        input_df,
        question_col="question",
        ground_truth_col="ground_truth",
        contexts_col="retrieved_contexts",
        relevance_threshold=0.2,
        ndcg_k=ndcg_k,
    )

    # 전략 그룹화 요약
    summary = summarize_retrieval(evaluated, group_col="strategy")

    # 예시 이미지 형태의 멋진 터미널 표 출력
    print_summary_table(summary)

    # 결과 파일 저장
    evaluated.to_csv(output_path, index=False, encoding="utf-8-sig")
    summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print(f"🎉 상세 결과 저장 완료: {output_path}")
    print(f"🎉 전략 요약 저장 완료: {summary_path}")
    return 0

In [14]:
import pandas as pd

# 1. 뼈대 데이터 가져오기
base_df = make_default_ground_truth_dataframe()

# 2. 이미지와 동일하게 6개의 실험 전략 데이터셋 가상 생성 및 병합
strategies = [
    "A-300 Vector", "B-500 Vector", "C-800 Vector",
    "A-300 Hybrid", "B-500 Hybrid", "C-800 Hybrid"
]

final_rows = []
for idx, row in base_df.iterrows():
    # 실험 편차 시뮬레이션을 위해 전략별로 다른 문맥 주입
    for strategy in strategies:
        new_row = row.to_dict()
        new_row["strategy"] = strategy

        # 임시로 정답 텍스트를 변형하여 유사도 편차 유도
        if "A-300" in strategy:
            new_row["retrieved_contexts"] = row["ground_truth"]  # 고품질 검색 결과
        else:
            new_row["retrieved_contexts"] = "고려대학교 관련 임시 검색 문서 조각 내용입니다. " + row["ground_truth"][:20]

        final_rows.append(new_row)

mock_df = pd.DataFrame(final_rows)
mock_df.to_csv("multi_strategy_rag_data.csv", index=False, encoding="utf-8-sig")

# 3. k=3 조건으로 최종 수립된 run_eval 함수 실행!
run_eval("multi_strategy_rag_data.csv", ndcg_k=3)


  RAG Retrieval 전략별 평가 결과 요약 (B모드 / 토큰 overlap)
  전략                 context_recall  context_precision                MRR               nDCG
────────────────────────────────────────────────────────────────────────────────────────────
  A-300 Hybrid             1.0000 *           1.0000 *           1.0000 *           1.0000 *
  A-300 Vector             1.0000 *           1.0000 *           1.0000 *           1.0000 *
  B-500 Hybrid             0.3908             1.0000 *           1.0000 *           1.0000 *
  B-500 Vector             0.3908             1.0000 *           1.0000 *           1.0000 *
  C-800 Hybrid             0.3908             1.0000 *           1.0000 *           1.0000 *
  C-800 Vector             0.3908             1.0000 *           1.0000 *           1.0000 *
────────────────────────────────────────────────────────────────────────────────────────────
  * 각 지표 최고값

🎉 상세 결과 저장 완료: retrieval_eval_result.csv
🎉 전략 요약 저장 완료: retrieval_eval_summary.csv


0

In [16]:
# 파일명을 multi_strategy_rag_data.csv로 지정하여 실행
run_eval("multi_strategy_rag_data.csv", ndcg_k=3)


  RAG Retrieval 전략별 평가 결과 요약 (B모드 / 토큰 overlap)
  전략                 context_recall  context_precision                MRR               nDCG
────────────────────────────────────────────────────────────────────────────────────────────
  A-300 Hybrid             1.0000 *           1.0000 *           1.0000 *           1.0000 *
  A-300 Vector             1.0000 *           1.0000 *           1.0000 *           1.0000 *
  B-500 Hybrid             0.3908             1.0000 *           1.0000 *           1.0000 *
  B-500 Vector             0.3908             1.0000 *           1.0000 *           1.0000 *
  C-800 Hybrid             0.3908             1.0000 *           1.0000 *           1.0000 *
  C-800 Vector             0.3908             1.0000 *           1.0000 *           1.0000 *
────────────────────────────────────────────────────────────────────────────────────────────
  * 각 지표 최고값

🎉 상세 결과 저장 완료: retrieval_eval_result.csv
🎉 전략 요약 저장 완료: retrieval_eval_summary.csv


0

In [17]:
def run_eval(input_file: str, ndcg_k: Optional[int] = 3, relevance_threshold: float = 0.2) -> int:
    """
    코랩 셀에서 바로 실행하기 위한 진입점입니다.
    relevance_threshold 값을 조절하여 변별력을 테스트할 수 있습니다.
    """
    import pandas as pd

    output_path = "retrieval_eval_result.csv"
    summary_path = "retrieval_eval_summary.csv"

    # 데이터 로드
    input_df = _load_input(input_file, 0)

    # 평가 수행 (relevance_threshold 반영)
    evaluated = evaluate_retrieval_dataframe(
        input_df,
        question_col="question",
        ground_truth_col="ground_truth",
        contexts_col="retrieved_contexts",
        relevance_threshold=relevance_threshold,  # 👈 넘겨받은 문턱 점수 적용
        ndcg_k=ndcg_k,
    )

    # 전략 그룹화 요약
    summary = summarize_retrieval(evaluated, group_col="strategy")

    # 터미널 표 출력 (어떤 문턱 점수인지 제목에 표시)
    print(f"\n[실험 조건] 내장 평가 기준 문턱 점수 (Relevance Threshold): {relevance_threshold}")
    print_summary_table(summary)

    # 결과 파일 저장
    evaluated.to_csv(output_path, index=False, encoding="utf-8-sig")
    summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print(f"🎉 상세 결과 저장 완료: {output_path}")
    print(f"🎉 전략 요약 저장 완료: {summary_path}")
    return 0

In [18]:
# 1단계: 기준을 0.4로 높여서 모든 전략 일괄 채점
run_eval("multi_strategy_rag_data.csv", ndcg_k=3, relevance_threshold=0.4)

print("\n" + "="*50 + "\n")

# 2단계: 기준을 0.5로 더 높여서 모든 전략 일괄 채점
run_eval("multi_strategy_rag_data.csv", ndcg_k=3, relevance_threshold=0.5)


[실험 조건] 내장 평가 기준 문턱 점수 (Relevance Threshold): 0.4

  RAG Retrieval 전략별 평가 결과 요약 (B모드 / 토큰 overlap)
  전략                 context_recall  context_precision                MRR               nDCG
────────────────────────────────────────────────────────────────────────────────────────────
  A-300 Hybrid             1.0000 *           1.0000 *           1.0000 *           1.0000 *
  A-300 Vector             1.0000 *           1.0000 *           1.0000 *           1.0000 *
  B-500 Hybrid             0.3908             0.3000             0.3000             1.0000 *
  B-500 Vector             0.3908             0.3000             0.3000             1.0000 *
  C-800 Hybrid             0.3908             0.3000             0.3000             1.0000 *
  C-800 Vector             0.3908             0.3000             0.3000             1.0000 *
────────────────────────────────────────────────────────────────────────────────────────────
  * 각 지표 최고값

🎉 상세 결과 저장 완료: retrieval_eval_result.csv
🎉 전략 요약 

0